# Modelo: Random

In [1]:
import os, random
import pandas as pd
import sys; sys.path.append('..')
from src.evaluation import temporal_train_test_split, evaluate_model

MODEL_NAME = 'random'
RESULTS_DIR = f'../results/{MODEL_NAME}'

## Definición del modelo

In [2]:
class RandomRecommender:
    def __init__(self, seed=42):
        self._rng = random.Random(seed)
        self._all_items = []

    def fit(self, train_reviews):
        self._all_items = train_reviews['business_id'].unique().tolist()
        return self

    def recommend(self, user_id, train_reviews, top_k=10):
        seen = set(train_reviews[train_reviews['user_id'] == user_id]['business_id'])
        candidates = [i for i in self._all_items if i not in seen]
        k = min(top_k, len(candidates))
        return self._rng.sample(candidates, k) if k > 0 else []

## Datos

In [3]:
reviews = pd.read_csv('../data/processed/reviews.csv', parse_dates=['date'])
train_reviews, test_reviews = temporal_train_test_split(reviews, test_fraction=0.2)
print(f'Train: {len(train_reviews)} | Test: {len(test_reviews)}')

Train: 83256 reviews | Test: 17191 reviews
Train: 83256 | Test: 17191


## Entrenamiento y evaluación

In [4]:
model = RandomRecommender(seed=42).fit(train_reviews)

metrics = evaluate_model(
    lambda uid, top_k: model.recommend(uid, train_reviews, top_k),
    test_reviews, train_reviews, k_values=[5, 10, 20]
)
print(metrics.round(4))

    precision  recall    ndcg
K                            
5      0.0011  0.0038  0.0024
10     0.0012  0.0077  0.0038
20     0.0012  0.0150  0.0059


## Guardar resultados

In [5]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics.to_csv(f'{RESULTS_DIR}/metrics.csv')
print(f'Saved -> results/{MODEL_NAME}/metrics.csv')

Saved -> results/random/metrics.csv
